# R3GRID — 조각 3 (네 조각을 런타임 하나씩)

라운드 3 학생만 은닉층·학습률·배치·에포크 54조합으로 다시 학습하고, 앞 정책에 쌓아 한 시간 SUMO로 잽니다. 새 굴림은 없습니다.
24상황으로 거른 뒤 상위 3개를 84상황에서 라운드 3 정책과 비교합니다. 방법과 읽는 법은 `results/explore/r3grid_20261105/사전등록.md`에 고정했습니다.

`shard0`~`shard3`을 **각각 런타임 하나씩** 씁니다. **동시에 시작해도 됩니다.** 조각 0이 점검(약 10분)과 학생 학습(약 30분, 어림)을 끝내면 나머지가 이어서 돕니다.

1. 런타임 유형 **CPU + 높은 RAM(CPU 8개)** 을 고르고 **런타임 → 모두 실행**을 누르세요. GPU는 필요 없습니다.
2. 이 노트북 하나가 끝까지 돕니다: 점검 → 학생 학습 → 거르기(3,816판) → 확인(1,344판). 합쳐 약 5~6시간(어림).
3. 끊기면 그 노트북만 다시 **모두 실행**하세요. 끝난 부분은 건너뜁니다. 다른 조각은 기다립니다.
4. 네 조각이 모두 끝나면 `R3GRID_judge`를 실행하고 "끝났어"라고 알려 주세요. 결과: Drive `FutureTraffic/explore_tune/r3grid/`.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json, time

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/FutureTraffic")
ARCHIVE = DRIVE / "FutureTraffic_R3GRID.zip"
WORKSPACE = Path("/content/FutureTraffic_R3GRID")
N3 = DRIVE / "explore_nn3"                             # 자료·모델·저장 결과(읽기만, 해시 대조)
OUT = DRIVE / "explore_tune" / "r3grid"
SHARD = 3
MIN_CPUS = 8
for path in (ARCHIVE, N3 / "prep" / "start.json", N3 / "r3" / "students" / "done.json", N3 / "r3" / "eval" / "runs" / "recent_s10_m0", N3 / "r2" / "data" / "runs" / "DATA", OUT / "status"):
    assert path.exists(), f"필요한 파일·폴더가 없습니다: {path}"
CPUS = os.cpu_count() or 1
print("CPU 개수:", CPUS, "| 조각:", SHARD, "| 결과 폴더:", OUT)
assert CPUS >= MIN_CPUS, (f"CPU가 {CPUS}개입니다. 런타임 → 런타임 유형 변경에서 CPU와 '높은 RAM'을 고른 뒤 다시 실행하세요.")

In [ ]:
# ZIP 전체의 파일 목록·SHA-256을 확인한 뒤 검증된 전용 추출기를 사용합니다.
import hashlib, stat, types, zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) and names.count("manifest.json") == 1, "ZIP 파일 목록 오류"
    package_manifest = json.loads(archive.read("manifest.json"))
    assert package_manifest.get("kind") == "WEIGHT1_PACKAGE_V1", "검증할 수 없는 ZIP 형식입니다"
    assert package_manifest.get("experiment") == "R3GRID_V1", "R3GRID 전용 ZIP이 아닙니다"
    assert package_manifest.get("source_models_included") is False, "모델·결과가 든 묶음은 쓰지 않습니다"
    expected = package_manifest["files"]
    assert expected and "manifest.json" not in expected and set(names) == set(expected) | {"manifest.json"}, "ZIP과 manifest 목록 불일치"
    for info in archive.infolist():
        name, path = info.filename, PurePosixPath(info.filename)
        assert path.parts and not path.is_absolute() and ".." not in path.parts and path.as_posix() == name, "잘못된 ZIP 경로"
        assert "\\" not in name and not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16), "잘못된 ZIP 파일 형식"
        if name != "manifest.json":
            assert hashlib.sha256(archive.read(name)).hexdigest() == expected[name], f"SHA-256 불일치: {name}"
    source_code = archive.read("src/explore_weight_package.py").decode("utf-8")

bootstrap = types.ModuleType("r3grid_verified_package")
bootstrap.__file__ = str(WORKSPACE / "src/explore_weight_package.py")
exec(compile(source_code, bootstrap.__file__, "exec"), bootstrap.__dict__)
package_manifest = bootstrap.extract_verified(ARCHIVE, WORKSPACE)
print(f"검증·추출 완료: {len(package_manifest['files'])}개 파일 → {WORKSPACE} | 잠금 {package_manifest['lock_sha256'][:12]} "
      f"| PLATOON0 설정 {package_manifest['platoon0_config_sha256'][:12]} | 측정 계획 {package_manifest['plan_sha256'][:12]}")

In [ ]:
import subprocess, importlib.metadata

os.environ["FOCUSED_RULESET"] = "pet1"
os.environ["FOCUSED_NORTH_EXT"] = "240"
# ARENA3 심판 기준선과 같은 평가 지문이 되도록 같은 패키지 버전을 씁니다.
EXPECTED = {"numpy": "2.1.3", "torch": "2.11.0+cpu", "stable-baselines3": "2.9.0",
            "sumo-rl": "1.4.5", "eclipse-sumo": "1.27.1"}

def installed_versions():
    versions = {}
    for name in EXPECTED:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

before = installed_versions()
if before == EXPECTED:
    print("이전 실험과 패키지 버전이 같아 재설치를 건너뜁니다.")
else:
    print("이전 실험과 같은 패키지 버전을 준비합니다:", before)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "stable-baselines3==2.9.0", "eclipse-sumo==1.27.1"],
                   cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cpu", "--index-url", "https://download.pytorch.org/whl/cpu"],
                   cwd=WORKSPACE, check=True)
after = installed_versions()
assert after == EXPECTED, f"패키지 버전이 이전 실험과 다릅니다(평가 지문이 달라져 기준선과 비교할 수 없어 멈춥니다): {after}"
print("실행 패키지 검증 통과:", after)

In [ ]:
os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE / "src"))
import arena3_colab as colab
from IPython.display import display, Markdown

colab.run_live(["-m", "unittest", "discover", "-s", "tests", "-p", "test_explore_r3grid.py"], WORKSPACE)
print("실행 전 검사(SUMO 없이) 통과")

In [ ]:
# 실행 전 점검: 조각 0이 돌리고(약 10분), 다른 조각은 결과를 기다립니다. 통과해야 본 실행으로 갑니다.
CHECK = OUT / "check"
result = CHECK / "check.json"
if SHARD == 0 and not (colab.read_json(result) or {}).get("ok"):
    colab.run_live(["src/explore_r3grid.py", "check", "--n3", str(N3), "--out", str(CHECK), "--work", "/content/rc"], WORKSPACE, check=False)
while not result.exists():
    print("조각 0의 점검을 기다립니다."); time.sleep(60)
r = colab.read_json(result)
assert r and r.get("ok"), f"점검 실패 — 본 실행을 하지 않습니다. 이 칸의 출력을 저에게 보내 주세요: {r}"
print("점검 통과 | 저장 결과와 같음:", r["same_as_stored"], "| 값이 있는 3층 학생이 바꾼 결정:", r["active_changed_decisions"])

In [ ]:
# 학생 → 거르기 → 확인. 백그라운드로 돌아 셀을 멈춰도 계속 돕니다. 끊기면 처음부터 다시 모두 실행하세요.
LOGS = OUT / "logs" / f"shard{SHARD}"

def describe(name):
    students = (OUT / "students" / "done.json").exists()
    stages = [x for x in ("screen", "final") if (OUT / "status" / f"{x}_shard{SHARD}.json").exists()]
    return f"학생 끝: {students}, 끝난 단계 {stages} | " + " / ".join(colab.tail(LOGS / f"{name}.log", 1))

pids = colab.launch({f"r3grid_shard{SHARD}": ["src/explore_r3grid.py", "run", "--n3", str(N3), "--out", str(OUT), "--work", "/content/rg", "--shard", str(SHARD), "--shards", "4", "--workers", str(CPUS)]}, WORKSPACE, LOGS)
code = colab.watch(pids, LOGS, 300, describe)
assert code == 0, "오류가 났습니다. 위 로그를 저에게 보내 주세요(다시 실행하면 끝난 부분은 건너뜁니다)."
print("이 조각 끝. 네 조각이 모두 끝나면 R3GRID_judge를 실행하세요.")